In [1]:
# 1. Install Ultralytics and force NumPy to stay at version 1.x
# We use a single command to prevent them from overwriting each other
!pip install "ultralytics==8.3.40" "numpy<2.0" "matplotlib>=3.3.0" supervision

# 2. IMPORTANT: Verification
import numpy as np
import ultralytics
print(f"✅ NumPy Version: {np.__version__}")
print(f"✅ Ultralytics Version: {ultralytics.__version__}")

# If this prints NumPy 2.x, you must restart the kernel.
# If it prints 1.26.4, you are safe to proceed.

INFO: pip is looking at multiple versions of opencv-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 898.5/898.5 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/

In [2]:
# --- BLOCK 1: INSTALL & SETUP ---
import os
import yaml
from ultralytics import YOLO

# 1. Install dependencies & Remove conflicts (Ray/WandB causes crashes)
os.system('pip install "ultralytics==8.3.40" "numpy<2.0" "matplotlib>=3.3.0" supervision')
os.system('pip uninstall -y ray wandb') 

# 2. Setup Dataset Config
INPUT_DIR = '/kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset'
config = {
    'path': INPUT_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {
        0: 'animal', 1: 'autorickshaw', 2: 'bicycle', 3: 'bus', 4: 'car',
        5: 'caravan', 6: 'motorcycle', 7: 'person', 8: 'rider', 9: 'traffic light',
        10: 'traffic sign', 11: 'trailer', 12: 'train', 13: 'truck', 14: 'vehicle fallback'
    }
}

with open('/kaggle/working/idd_fixed.yaml', 'w') as f:
    yaml.dump(config, f)

print("✅ Setup complete. Environment ready.")


# --- BLOCK 2: CUSTOM PROGRESS TRACKER ---
def on_train_epoch_end(trainer):
    """
    This function runs automatically at the end of every epoch.
    It prints a clear text-based progress bar for the logs.
    """
    current_epoch = trainer.epoch + 1
    total_epochs = trainer.epochs
    progress = (current_epoch / total_epochs) * 100
    
    # Create a simple visual bar for the logs
    bar_length = 20
    filled_length = int(bar_length * current_epoch // total_epochs)
    bar = '█' * filled_length + '-' * (bar_length - filled_length)
    
    print(f"\n========================================================")
    print(f"🚀 PROGRESS: [{bar}] {progress:.1f}%")
    print(f"   Epoch: {current_epoch}/{total_epochs}")
    print(f"   GPU Memory: {trainer.gpu_memory_used:.2f} GB (approx)")
    print(f"========================================================\n")


# --- BLOCK 3: TRAINING ---
if __name__ == '__main__':
    print("🚀 Loading YOLOv8m model...")
    model = YOLO('yolov8m.pt')

    # Attach our custom progress tracker to the model
    model.add_callback("on_train_epoch_end", on_train_epoch_end)

    print("🚀 Starting Multi-GPU Training...")
    results = model.train(
        data='/kaggle/working/idd_fixed.yaml',
        epochs=50,
        imgsz=640,
        batch=32,            
        device=[0, 1],       
        project='/kaggle/working/runs',
        name='idd_yolov8_training',
        exist_ok=True,
        verbose=True, # Keeps standard logs too
        plots=True
    )
    
    print(f"✅ Training Complete! Best model: {results.save_dir}/weights/best.pt")

Found existing installation: ray 2.51.1
Uninstalling ray-2.51.1:
  Successfully uninstalled ray-2.51.1
Found existing installation: wandb 0.21.0
Uninstalling wandb-0.21.0:
  Successfully uninstalled wandb-0.21.0
✅ Setup complete. Environment ready.
🚀 Loading YOLOv8m model...


100%|██████████| 49.7M/49.7M [00:00<00:00, 148MB/s]


🚀 Starting Multi-GPU Training...
New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
                                                       CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8m.pt, data=/kaggle/working/idd_fixed.yaml, epochs=50, time=None, patience=100, batch=32, imgsz=640, save=True, save_period=-1, cache=False, device=[0, 1], workers=8, project=/kaggle/working/runs, name=idd_yolov8_training, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_s

100%|██████████| 755k/755k [00:00<00:00, 17.3MB/s]
E0000 00:00:1765082490.103840      21 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765082490.142131      21 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 132/132 [00:49<00:00,  2.66it/s]


                   all       4196      49838      0.747      0.506      0.542      0.371
                animal        219        753      0.746      0.352        0.4       0.21
          autorickshaw       1436       3205      0.881      0.695      0.774      0.578
               bicycle        264        301      0.828      0.522      0.575      0.366
                   bus       1061       1794       0.88      0.676      0.762      0.601
                   car       2582       8793      0.885      0.624      0.716      0.525
               caravan         18         18      0.432      0.722      0.478      0.466
            motorcycle       2778      10059      0.871      0.641      0.717      0.457
                person       2105       8863      0.843       0.46      0.565      0.333
                 rider       2454       9444      0.863      0.528      0.626      0.391
         traffic light        157        370      0.769      0.405       0.49      0.278
          traffic sig

/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1
/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1


Speed: 0.1ms preprocess, 5.7ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /kaggle/working/runs/idd_yolov8_training


AttributeError: 'NoneType' object has no attribute 'save_dir'